##Load Dataset

In [2]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("Vehicular_Crash_Data.csv", low_memory=False)


df = df_raw.copy()

print("Raw rows:", len(df_raw))
print("Raw columns:", len(df_raw.columns))

Raw rows: 354151
Raw columns: 66


##Initial data review

In [3]:
df_raw.head()

,X,Y,CRIMEID,CCN,REPORTDATE,ROUTEID,MEASURE,OFFSET,STREETSEGID,ROADWAYSEGID,...,MAR_ID,BLOCKKEY,SUBBLOCKKEY,CORRIDORID,NEARESTINTKEY,MAJORINJURIESOTHER,MINORINJURIESOTHER,UNKNOWNINJURIESOTHER,FATALOTHER,OBJECTID
0,-8.568062e+06,4.701533e+06,23417571,11181422,2011/12/11 05:00:00+00,13063502,1764.39,24.83,400.0,670.0,...,294547,787e8c163fae3e7b5902860b80d1a930,787e8c163fae3e7b5902860b80d1a930,13063502_2,aa8b81a790ddc22ed4d38eeca15a2829,NaN,NaN,NaN,NaN,516601809
1,-8.563973e+06,4.704270e+06,23417777,11180570,2011/12/10 03:45:00+00,13015342,1343.15,28.39,7570.0,16963.0,...,15464,bb64cf3216e26cf2b8c48247dcd6deb0,8b906dcc29445426c3de263b7fa0a864,Blockkey Not Found on Corridor,663a9729bcdd1dac58b7c7dac9a72749,NaN,NaN,NaN,NaN,516601810
2,-8.571420e+06,4.713979e+06,23419107,10119565,2010/08/03 04:00:00+00,12018782,282.49,21.41,275.0,227.0,...,4982,a98da7dd5562fcef6d3957fa6907914f,a98da7dd5562fcef6d3957fa6907914f,12018782_2,301cd2761de06819b28db980eebf2d02,NaN,NaN,NaN,NaN,516601811
3,-8.568250e+06,4.701659e+06,23419243,10119317,2010/08/19 04:00:00+00,13002502,763.19,0.04,8754.0,9385.0,...,48456,0eae799caf0901595405a3978a5ad605,5e3742b7dbbaf240bff6b29f01e7993d,13002502_2,82630c5f8b387794a2730460bd10e647,NaN,NaN,NaN,NaN,516601812
4,-8.571175e+06,4.706062e+06,23419706,10119693,2010/08/20 04:00:00+00,12000702,125.48,19.28,2180.0,17894.0,...,76087,9ed2dfe38138d5e322701362a0b8c84a,81ac58fb8077653ef742f9d9ce79b7b1,12000702_1,4a99080d26e79d35b2ca791b5ea44624,NaN,NaN,NaN,NaN,516601813


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354151 entries, 0 to 354150
Data columns (total 66 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   X                           354151 non-null  float64
 1   Y                           354151 non-null  float64
 2   CRIMEID                     354151 non-null  int64  
 3   CCN                         354151 non-null  object 
 4   REPORTDATE                  352772 non-null  object 
 5   ROUTEID                     354151 non-null  object 
 6   MEASURE                     354151 non-null  float64
 7   OFFSET                      354151 non-null  float64
 8   STREETSEGID                 215903 non-null  float64
 9   ROADWAYSEGID                215903 non-null  float64
 10  FROMDATE                    354074 non-null  object 
 11  TODATE                      0 non-null       float64
 12  ADDRESS                     354069 non-null  object 
 13  LATITUDE      

In [5]:
types_table = pd.DataFrame({
    "Column": df_raw.columns,
    "Data type": df_raw.dtypes.astype(str).values,
    "Non-null values": df_raw.notna().sum().values,
    "Missing values": df_raw.isna().sum().values
})

types_table

,Column,Data type,Non-null values,Missing values
0,X,float64,354151,0
1,Y,float64,354151,0
2,CRIMEID,int64,354151,0
3,CCN,object,354151,0
4,REPORTDATE,object,352772,1379
...,...,...,...,...
61,MAJORINJURIESOTHER,float64,100599,253552
62,MINORINJURIESOTHER,float64,100599,253552
63,UNKNOWNINJURIESOTHER,float64,100599,253552
64,FATALOTHER,float64,100599,253552


##Missing values and data quality check

In [6]:
missing = (
    df_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

missing["Missing %"] = (
    missing["Missing Values"] / len(df_raw) * 100
).round(2)

missing.head(15)

,Missing Values,Missing %
TODATE,354151,100.00
LOCATIONERROR,302838,85.51
MINORINJURIESOTHER,253552,71.59
MAJORINJURIESOTHER,253552,71.59
FATALOTHER,253552,71.59
UNKNOWNINJURIESOTHER,253552,71.59
LASTUPDATEDATE,225127,63.57
MPDGEOY,221787,62.62
MPDGEOX,221787,62.62
STREETSEGID,138248,39.04


In [7]:
fully_empty_rows = df_raw.isna().all(axis=1).sum()

empty_columns = df_raw.columns[
    df_raw.isna().all(axis=0)
].tolist()

print("Fully empty rows:", fully_empty_rows)
print("Fully empty columns:", len(empty_columns))
print("Empty column names:", empty_columns)

Fully empty rows: 0
Fully empty columns: 1
Empty column names: ['TODATE']


In [8]:
df_raw["WARD"].value_counts(dropna=False)

,count
WARD,
Ward 2,72440
Ward 5,55191
Ward 6,51189
Ward 7,48701
Ward 8,43506
Ward 1,31042
Ward 4,30369
Ward 3,20280
Null,1430


In [9]:
date_test = pd.to_datetime(
    df_raw["REPORTDATE"],
    errors="coerce",
    utc=True
)

print("Original REPORTDATE type:", df_raw["REPORTDATE"].dtype)

print(
    "Successfully parsed:",
    date_test.notna().sum()
)

print(
    "Missing / unparseable:",
    date_test.isna().sum()
)

print(
    "Earliest parsed date:",
    date_test.min()
)

print(
    "Latest parsed date:",
    date_test.max()
)

Original REPORTDATE type: object
Successfully parsed: 352772
Missing / unparseable: 1379
Earliest parsed date: 1825-11-08 11:57:02+00:00
Latest parsed date: 2026-09-14 21:32:00+00:00


In [10]:
year_distribution = (
    date_test.dt.year
    .value_counts(dropna=False)
    .sort_index()
)

year_distribution

,count
REPORTDATE,
1825.0,1
1826.0,1
1827.0,1
1900.0,46
1975.0,1
1976.0,1
1988.0,1
1989.0,4
1990.0,4


In [11]:
# Check coordinates outside the Washington, DC area
outside_dc = df[
    ~(
        df["LATITUDE"].between(38.75, 39.05)
        &
        df["LONGITUDE"].between(-77.20, -76.85)
    )
    &
    df["LATITUDE"].notna()
    &
    df["LONGITUDE"].notna()
]

print("Records with coordinates outside DC:", len(outside_dc))

outside_dc[
    [
        "CRIMEID",
        "LATITUDE",
        "LONGITUDE",
        "ADDRESS"
    ]
].head(20)

Records with coordinates outside DC: 2


,CRIMEID,LATITUDE,LONGITUDE,ADDRESS
112687,26719441,38.913063,76.997665,400 RHODE ISLAND AVE
127693,26930521,38.909001,77.012701,I395 AT 3RD ST TUNNEL NORTH BOUND


##Preprocessing

Clean dates:

In [12]:
# Parsing the original date field
df["REPORTDATE_PARSED"] = pd.to_datetime(
    df["REPORTDATE"],
    errors="coerce",
    utc=True
)

year = df["REPORTDATE_PARSED"].dt.year


valid_date = year.between(2008, 2026)

df["DATE_STATUS"] = np.where(
    valid_date,
    "Valid analysis date",
    "Invalid / outside analysis period"
)

# Keeping invalid dates as missing only in the cleaned analytical field
df["REPORTDATE_CLEAN"] = (
    df["REPORTDATE_PARSED"]
    .where(valid_date)
)

df["YEAR"] = (
    df["REPORTDATE_CLEAN"]
    .dt.year
)

df["MONTH"] = (
    df["REPORTDATE_CLEAN"]
    .dt.strftime("%Y-%m")
)

Standardize ward values:

In [13]:
def clean_ward(value):

    if pd.isna(value):
        return "Unknown / not assigned"

    value = str(value).strip()

    invalid_values = [
        "",
        "NULL",
        "UNKNOWN",
        "NAN",
        "NONE"
    ]

    if value.upper() in invalid_values:
        return "Unknown / not assigned"

    return value


df["WARD_CLEAN"] = (
    df["WARD"]
    .apply(clean_ward)
)

Convert relevant variables to numeric:

In [14]:
numeric_fields = [

    "TOTAL_VEHICLES",
    "TOTAL_BICYCLES",
    "TOTAL_PEDESTRIANS",

    "SPEEDING_INVOLVED",

    "FATAL_BICYCLIST",
    "FATAL_DRIVER",
    "FATAL_PEDESTRIAN",
    "FATALPASSENGER",
    "FATALOTHER",

    "MAJORINJURIES_BICYCLIST",
    "MAJORINJURIES_DRIVER",
    "MAJORINJURIES_PEDESTRIAN",
    "MAJORINJURIESPASSENGER",
    "MAJORINJURIESOTHER",

    "LATITUDE",
    "LONGITUDE"
]

for column in numeric_fields:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

Total Fatalities:

In [15]:
fatality_fields = [

    "FATAL_BICYCLIST",
    "FATAL_DRIVER",
    "FATAL_PEDESTRIAN",
    "FATALPASSENGER",
    "FATALOTHER"
]

df["TOTAL_FATALITIES"] = (
    df[fatality_fields]
    .fillna(0)
    .sum(axis=1)
)

Total Major Injuries:

In [16]:
major_injury_fields = [

    "MAJORINJURIES_BICYCLIST",
    "MAJORINJURIES_DRIVER",
    "MAJORINJURIES_PEDESTRIAN",
    "MAJORINJURIESPASSENGER",
    "MAJORINJURIESOTHER"
]

df["TOTAL_MAJOR_INJURIES"] = (
    df[major_injury_fields]
    .fillna(0)
    .sum(axis=1)
)

Dashboard indicators (to simplify filtering and visualization in the dashboard):

In [17]:
df["PEDESTRIAN_INVOLVED"] = np.where(
    df["TOTAL_PEDESTRIANS"].fillna(0) > 0,
    "Yes",
    "No"
)

df["BICYCLE_INVOLVED"] = np.where(
    df["TOTAL_BICYCLES"].fillna(0) > 0,
    "Yes",
    "No"
)

df["SPEEDING_FLAG"] = np.where(
    df["SPEEDING_INVOLVED"].fillna(0) > 0,
    "Yes",
    "No"
)

print("Preprocessing completed")

Preprocessing completed


Сoordinates outside the expected Washington, DC area:

In [18]:
coordinate_valid = (
    df["LATITUDE"].between(38.75, 39.05)
    &
    df["LONGITUDE"].between(-77.20, -76.85)
)

df["COORDINATE_STATUS"] = np.where(
    coordinate_valid,
    "Valid DC coordinate",
    "Outside expected DC area"
)

map_source = df[
    df["COORDINATE_STATUS"]
    == "Valid DC coordinate"
].copy()

##Validation after preprocessing

In [19]:
validation_summary = pd.DataFrame({

    "Check": [

        "Original records",
        "Records after preprocessing",

        "Original variables",
        "Variables after preprocessing",

        "Valid dates for time analysis",
        "Invalid / out-of-range dates",

        "Unknown / unassigned ward",

        "Completely empty records"
    ],

    "Value": [

        len(df_raw),
        len(df),

        len(df_raw.columns),
        len(df.columns),

        (
            df["DATE_STATUS"]
            == "Valid analysis date"
        ).sum(),

        (
            df["DATE_STATUS"]
            == "Invalid / outside analysis period"
        ).sum(),

        (
            df["WARD_CLEAN"]
            == "Unknown / not assigned"
        ).sum(),

        df.isna().all(axis=1).sum()
    ]
})

validation_summary

,Check,Value
0,Original records,354151
1,Records after preprocessing,354151
2,Original variables,66
3,Variables after preprocessing,78
4,Valid dates for time analysis,352406
5,Invalid / out-of-range dates,1745
6,Unknown / unassigned ward,1433
7,Completely empty records,0


In [20]:
df[
    [
        "REPORTDATE",
        "REPORTDATE_CLEAN",

        "WARD",
        "WARD_CLEAN",

        "TOTAL_FATALITIES",
        "TOTAL_MAJOR_INJURIES",

        "PEDESTRIAN_INVOLVED",
        "BICYCLE_INVOLVED",
        "SPEEDING_FLAG"
    ]
].head(10)

,REPORTDATE,REPORTDATE_CLEAN,WARD,WARD_CLEAN,TOTAL_FATALITIES,TOTAL_MAJOR_INJURIES,PEDESTRIAN_INVOLVED,BICYCLE_INVOLVED,SPEEDING_FLAG
0,2011/12/11 05:00:00+00,2011-12-11 05:00:00+00:00,Ward 8,Ward 8,0.0,1.0,No,No,No
1,2011/12/10 03:45:00+00,2011-12-10 03:45:00+00:00,Ward 7,Ward 7,0.0,0.0,No,No,No
2,2010/08/03 04:00:00+00,2010-08-03 04:00:00+00:00,Ward 5,Ward 5,0.0,0.0,No,No,No
3,2010/08/19 04:00:00+00,2010-08-19 04:00:00+00:00,Ward 8,Ward 8,0.0,1.0,Yes,No,No
4,2010/08/20 04:00:00+00,2010-08-20 04:00:00+00:00,Ward 6,Ward 6,0.0,0.0,No,No,No
5,2011/12/12 06:15:00+00,2011-12-12 06:15:00+00:00,Ward 2,Ward 2,0.0,0.0,No,No,No
6,2011/12/13 05:00:00+00,2011-12-13 05:00:00+00:00,Ward 8,Ward 8,0.0,1.0,No,No,No
7,2011/12/13 05:00:00+00,2011-12-13 05:00:00+00:00,Ward 1,Ward 1,0.0,1.0,No,No,No
8,2011/12/13 05:00:00+00,2011-12-13 05:00:00+00:00,Ward 8,Ward 8,0.0,0.0,No,No,No
9,2011/12/13 05:00:00+00,2011-12-13 05:00:00+00:00,Ward 8,Ward 8,0.0,3.0,No,No,No


In [21]:
df["WARD_CLEAN"].value_counts(dropna=False)

,count
WARD_CLEAN,
Ward 2,72440
Ward 5,55191
Ward 6,51189
Ward 7,48701
Ward 8,43506
Ward 1,31042
Ward 4,30369
Ward 3,20280
Unknown / not assigned,1433


In [22]:
print(
    "Total fatalities:",
    int(df["TOTAL_FATALITIES"].sum())
)

print(
    "Total major injuries:",
    int(df["TOTAL_MAJOR_INJURIES"].sum())
)

print()

print("Pedestrian involvement:")
print(
    df["PEDESTRIAN_INVOLVED"]
    .value_counts()
)

print()

print("Bicycle involvement:")
print(
    df["BICYCLE_INVOLVED"]
    .value_counts()
)

print()

print("Speeding involvement:")
print(
    df["SPEEDING_FLAG"]
    .value_counts()
)

Total fatalities: 714
Total major injuries: 28667

Pedestrian involvement:
PEDESTRIAN_INVOLVED
No     336770
Yes     17381
Name: count, dtype: int64

Bicycle involvement:
BICYCLE_INVOLVED
No     346494
Yes      7657
Name: count, dtype: int64

Speeding involvement:
SPEEDING_FLAG
No     346445
Yes      7706
Name: count, dtype: int64


##Export

In [23]:
output_file = (
    "/content/"
    "Vehicular_Crash_Data_Cleaned.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

print(
    "Exported rows:",
    len(df)
)

print(
    "Exported columns:",
    len(df.columns)
)

Saved: /content/Vehicular_Crash_Data_Cleaned.csv
Exported rows: 354151
Exported columns: 78


In [24]:
from google.colab import files

files.download(
    "/content/Vehicular_Crash_Data_Cleaned.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Data for web prototype

In [25]:
import json
web_df = df.copy()

# Year label
web_df["YEAR_WEB"] = (
    web_df["YEAR"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown / invalid date")
)

# Month label
web_df["MONTH_WEB"] = (
    web_df["REPORTDATE_CLEAN"]
    .dt.strftime("%Y-%m")
)

# Round coordinates for map aggregation
web_df["LAT_GRID"] = (
    web_df["LATITUDE"]
    .round(2)
)

web_df["LON_GRID"] = (
    web_df["LONGITUDE"]
    .round(2)
)

Dashboard data:

In [26]:
summary_data = (
    web_df
    .groupby(
        [
            "YEAR_WEB",
            "MONTH_WEB",
            "WARD_CLEAN",
            "SPEEDING_FLAG",
            "PEDESTRIAN_INVOLVED",
            "BICYCLE_INVOLVED"
        ],
        dropna=False
    )
    .agg(
        crashes=("CRIMEID", "count"),
        fatalities=("TOTAL_FATALITIES", "sum"),
        major_injuries=("TOTAL_MAJOR_INJURIES", "sum")
    )
    .reset_index()
)

summary_data.head()

,YEAR_WEB,MONTH_WEB,WARD_CLEAN,SPEEDING_FLAG,PEDESTRIAN_INVOLVED,BICYCLE_INVOLVED,crashes,fatalities,major_injuries
0,2008,2008-01,Ward 2,No,No,No,2,0.0,0.0
1,2008,2008-01,Ward 3,No,No,No,1,0.0,0.0
2,2008,2008-01,Ward 4,No,No,No,1,0.0,0.0
3,2008,2008-01,Ward 6,No,No,No,1,0.0,0.0
4,2008,2008-01,Ward 8,No,No,No,1,0.0,0.0


Map data:

In [27]:
map_data = (
    web_df[
        web_df["LAT_GRID"].notna()
        & web_df["LON_GRID"].notna()
    ]
    .groupby(
        [
            "YEAR_WEB",
            "WARD_CLEAN",
            "SPEEDING_FLAG",
            "PEDESTRIAN_INVOLVED",
            "BICYCLE_INVOLVED",
            "LAT_GRID",
            "LON_GRID"
        ]
    )
    .size()
    .reset_index(name="crashes")
)

map_data.head()

,YEAR_WEB,WARD_CLEAN,SPEEDING_FLAG,PEDESTRIAN_INVOLVED,BICYCLE_INVOLVED,LAT_GRID,LON_GRID,crashes
0,2008,Unknown / not assigned,No,No,No,38.96,-77.00,1
1,2008,Unknown / not assigned,Yes,No,No,38.87,-76.97,1
2,2008,Unknown / not assigned,Yes,No,No,38.96,-77.00,1
3,2008,Ward 1,No,No,No,38.91,-77.02,1
4,2008,Ward 1,No,No,No,38.92,-77.05,3


In [28]:
crashlens_data = {
    "metadata": {
        "source_records": int(len(df)),
        "source_columns": int(len(df.columns))
    },

    "summary": summary_data.where(
        pd.notna(summary_data),
        None
    ).to_dict(orient="records"),

    "map": map_data.where(
        pd.notna(map_data),
        None
    ).to_dict(orient="records")
}


output_file = "/content/crashes.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        crashlens_data,
        f,
        ensure_ascii=False
    )


print("Saved:", output_file)
print("Summary rows:", len(summary_data))
print("Map rows:", len(map_data))

Saved: /content/crashes.json
Summary rows: 6890
Map rows: 13050


In [29]:
files.download(
    "/content/crashes.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>